# Quantum Templates

Pre-defined arquitectures to build Ansatz (Layers) and Embeddings

In [ ]:
import pennylane as qml
from pennylane import numpy as np

dev_template = qml.device("default.qubit", wires=3)

@qml.qnode(dev_template)
def template_circuit(data, weights):
    # EMBEDDING 
    # We map classical data into rotation angles
    qml.AngleEmbedding(features=data, wires=range(3), rotation='X')
    
    # LAYERS 
    qml.StronglyEntanglingLayers(weights=weights, wires=range(3))
    
    return qml.expval(qml.PauliZ(0))

# Classical input data (3 features)
data = np.array([0.1, 0.5, 0.9], requires_grad=False)


# We ask the template for the required matrix shape, then use numpy to fill it
num_layers = 2
weights_shape = qml.StronglyEntanglingLayers.shape(n_layers=num_layers, n_wires=3)
# weights_shape will be a tuple like (2, 3, 3)
weights = np.random.random(weights_shape)

print("--- Result of the Template Circuit ---")
print(f"Output: {template_circuit(data, weights):.4f}")

print("\n--- Circuit Visualization ---")
print(qml.draw(template_circuit)(data, weights))

--- Result of the Template Circuit ---
Output: 0.0393

--- Circuit Visualization ---
0: ─╭AngleEmbedding(M0)─╭StronglyEntanglingLayers(M1)─┤  <Z>
1: ─├AngleEmbedding(M0)─├StronglyEntanglingLayers(M1)─┤     
2: ─╰AngleEmbedding(M0)─╰StronglyEntanglingLayers(M1)─┤     

M0 = 
[0.1 0.5 0.9]
M1 = 
[[[0.92279145 0.97419146 0.04132641]
  [0.5325633  0.76259658 0.3846974 ]
  [0.72962534 0.83318484 0.58250231]]

 [[0.83592676 0.82960821 0.61765921]
  [0.26664825 0.06636358 0.47715168]
  [0.35794583 0.3419134  0.6206503 ]]]


# Dynamic Quantum Circuits

Measure a qubit in the middle of a process to decide which gate to use

Only available in simulators and some type of hardware


**Deferred Measurement Principle:**

You can make any intermediate measurement at the end of the circuit if you use a auxiliar qubit to store the information

In [ ]:
import pennylane as qml
from pennylane import numpy as np

# NOT specify the number of wires, we allow default.qubit 
# to dynamically allocate auxiliary wires needed for the Deferred Measurement Principle.
dev_dynamic = qml.device("default.qubit")

@qml.qnode(dev_dynamic)
def dynamic_circuit(angle):
    # q0 in superposition
    qml.Hadamard(wires=0)
    
    # Mid-circuit measurement
    # Behind the scenes, PennyLane will use a hidden wire to store this result
    m_0 = qml.measure(0)
    
    # Conditional Logic
    # If m_0 measured 1, apply a rotation to qubit 1. 
    qml.cond(m_0 == 1, qml.RY)(angle, wires=1)
    
    # Another conditional example
    qml.cond(m_0 == 0, qml.PauliX)(wires=1)

    # We explicitly tell it to return probabilities only for wires 0 and 1
    return qml.probs(wires=[0, 1])

# Run the circuit
angle = np.pi / 4
print("--- Dynamic Circuit Results (Probabilities) ---")
print(f"Probabilities[|00>, |01>, |10>, |11>]: {dynamic_circuit(angle)}")

# Draw the circuit 
print("\n--- Dynamic Circuit Drawing ---")
print(qml.draw(dynamic_circuit)(angle))

--- Dynamic Circuit Results (Probabilities) ---
Probabilities[|00>, |01>, |10>, |11>]: [0.        0.5       0.4267767 0.0732233]

--- Dynamic Circuit Drawing ---
0: ──H──┤↗├──────────────┤ ╭Probs
1: ──────║───RY(0.79)──X─┤ ╰Probs
         ╚═══╩═════════╝         
